# Yen's K-Shortest Paths

Dijkstra's gave you the optimal route. Yen's gives you options.

Finding multiple ranked alternatives enables risk management, reveals underutilized paths, and allows comparison with historical operations.

We'll practice on the Movie graph first, then apply what we learn to logistics.

## What Yen's Does

Yen's finds the **k shortest paths** between two nodes.

Not just the best route—the best k routes, ranked by total cost. Imagine multiple paths from source to destination, colored from green (best) to yellow (5th best).

## Why Multiple Routes Matter

- **Capacity constraints**: The optimal route might be full
- **Risk management**: You need backup options if disruptions occur
- **Trade-off analysis**: Compare time vs. other factors
- **Historical comparison**: See if current routes are in the top k

## How It Works

Yen's builds on Dijkstra's:

1. Find the shortest path (Dijkstra's)
2. Systematically block edges and find next-best paths
3. Rank all discovered paths by total cost
4. Return the top k

It's guaranteed to find the k truly shortest paths.

## Yen's vs Dijkstra's

| Aspect | Dijkstra's | Yen's |
|--------|-----------|-------|
| Returns | Single best path | k best paths |
| Use case | "What's the optimal route?" | "What are my options?" |
| Internally | Single pass | Runs Dijkstra's multiple times |
| Performance | Faster | Scales with k |

For k = 1, Yen's behaves exactly like Dijkstra's.

## Setup: Connect to AGA

In [ ]:
import os
import pandas as pd
from datetime import timedelta
from graphdatascience.session import GdsSessions, AuraAPICredentials
from graphdatascience.session import DbmsConnectionInfo, SessionMemory
from dotenv import load_dotenv

pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

# Import neo4j driver Result for graph transformations
from neo4j import Result

# Import neo4j_viz for visualization
from neo4j_viz.gds import from_gds
from neo4j_viz.neo4j import from_neo4j

# Load environment variables
load_dotenv()

# Get Aura API credentials
client_id = os.getenv('AURA_CLIENT_ID')
client_secret = os.getenv('AURA_CLIENT_SECRET')
project_id = os.getenv('AURA_PROJECT_ID')  # set in .env only if your Aura account has multiple projects

# Get AuraDB connection info
uri = os.getenv('AURA_URI')
username = os.getenv('AURA_USERNAME')
password = os.getenv('AURA_PASSWORD')

# Create sessions manager
sessions = GdsSessions(
    api_credentials=AuraAPICredentials(client_id, client_secret, project_id=project_id)
)

print("Sessions manager created")

In [ ]:
# Create a GDS Session
gds = sessions.get_or_create(
    session_name="yens-practice",
    memory=SessionMemory.m_2GB,
    db_connection=DbmsConnectionInfo(
        uri=uri,
        username=username,
        password=password
    ),
    ttl=timedelta(minutes=30)
)

gds.verify_connectivity()
print(f"Connected to GDS Session: yens-practice")

## Project the Movie Graph

Let's project our Movies graph with actor collaborations.

In [ ]:
# Project actor collaborations
G, result = gds.graph.project(
    "actor-collaborations",
    """
    CALL {
        MATCH (a1:Actor)-[:ACTED_IN]->(m:Movie)<-[:ACTED_IN]-(a2:Actor)
        WHERE a1 <> a2
        WITH a1, a2, count(m) AS collaborations
        RETURN a1 AS source, a2 AS target, collaborations
    }
    RETURN gds.graph.project.remote(
        source,
        target,
        {relationshipProperties: {collaborations: collaborations}}
    )
    """
)

print(f"Projected graph: {G.name()}")
print(f"  Nodes: {G.node_count():,}")
print(f"  Relationships: {G.relationship_count():,}")

## Find Multiple Paths

Let's find the top 5 shortest paths between two actors.

One thing to note: Yen's uses `targetNode` (singular), not `targetNodes` like Dijkstra's. You can only find k paths to a single destination per call.

In [ ]:
# Get source and target node IDs
source_id = gds.find_node_id(["Actor"], {"name": "Shah Rukh Khan"})
target_id = gds.find_node_id(["Actor"], {"name": "Lee Jung-jae"})

print(f"Source: Shah Rukh Khan (ID: {source_id})")
print(f"Target: Lee Jung-jae (ID: {target_id})")

In [ ]:
# Run Yen's algorithm for top 5 paths
yens_result = gds.shortestPath.yens.stream(
    G,
    sourceNode=source_id,
    targetNode=target_id,
    k=5
)

print(yens_result)

Now you see not just the best path, but the top 4 alternatives as well.

## Understanding the Results

| Field | Type | Description |
|-------|------|-------------|
| index | Integer | 0-based rank of this path (0 = best) |
| totalCost | Float | Total cost from source to target |
| nodeIds | List | Node IDs along the path |
| costs | List | Accumulated cost at each step |
| path | Path | Cypher path entity for visualization |

Paths are returned in order of total cost—lowest first.

Let's see those paths in a more readable format.

In [ ]:
# Display the results
results = []
for idx, row in yens_result.iterrows():
    # Get actor names along the path
    path_names = gds.run_cypher("""
        UNWIND $nodeIds AS nodeId
        RETURN gds.util.asNode(nodeId).name AS name
    """, params={"nodeIds": list(row['nodeIds'])})
    
    results.append({
        'rank': row['index'] + 1,
        'hops': int(row['totalCost']),
        'path': ' -> '.join(path_names['name'].tolist())
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

### Visualize: All K-Shortest Paths

Let's visualize all 5 paths together to see how they diverge and converge.

In [ ]:
# Helper function: Execute query with Neo4j driver (for visualizations)
def execute_neo4j_query(query, params=None, database=os.getenv("AURA_DATABASE") or "neo4j"):
    from neo4j import GraphDatabase, RoutingControl
    with GraphDatabase.driver(uri, auth=(username, password)) as driver:
        driver.verify_connectivity()
        result = driver.execute_query(
            query,
            parameters_=params or {},
            database_=database,
            routing_=RoutingControl.READ,
            result_transformer_=Result.graph,
        )
    return result

# Helper function: Create visualization from Cypher query
def visualize_query(query, params=None, database=os.getenv("AURA_DATABASE") or "neo4j"):
    result = execute_neo4j_query(query, params, database)
    return from_neo4j(result)

# Helper function: Visualize a GDS graph projection
def visualize_projection(G):
    return from_gds(gds, G, max_node_count=100, max_relationship_count=100)

# Helper function: Visualize k shortest paths
def visualize_k_paths(yens_result, title="K-Shortest Paths"):
    """Visualize all k shortest paths from Yen's result."""
    # Collect all unique node IDs from all paths
    all_node_ids = set()
    for idx, row in yens_result.iterrows():
        all_node_ids.update(row['nodeIds'])
    
    # Get actor names
    names_df = gds.run_cypher("""
        UNWIND $nodeIds AS nodeId
        RETURN gds.util.asNode(nodeId).name AS name
    """, params={"nodeIds": list(all_node_ids)})
    names = list(set(names_df['name'].tolist()))
    
    # Visualize paths between these actors
    print(title)
    VG = visualize_query(f"""
        MATCH path = SHORTEST 1 (a1:Actor)-[:ACTED_IN]->(m:Movie)<-[:ACTED_IN]-(a2:Actor)
        WHERE a1.name IN {names} AND a2.name IN {names} AND a1 <> a2
        RETURN path
    """)
    return VG

print("Helper functions loaded")

In [ ]:
# Visualize all k shortest paths
VG = visualize_k_paths(yens_result, "Top 5 Shortest Paths: Shah Rukh Khan -> Lee Jung-jae")
VG.render()

You can see here that many of the paths run through the same nodes to converge on the same destination. 

## Parallel Relationships

Yen's respects **parallel relationships** between nodes.

If two actors collaborated on multiple movies, those are separate edges in the projection. Yen's can find paths that use different parallel edges between the same nodes.

This is useful when parallel relationships have different weights.

For example, in a logistics network, you may find that the same route costs more or less depending on the day. If you include parallel relationships in your projection, Yen's will only choose the cheapest of days.

However, bear this in mind if you have a network with thousands of parallel edges. Without weights, you will likely always get the same path for your topK.

## Weighted Paths

Now let's add collaboration counts as weights to find paths through the weakest collaborators.

Remember: Yen's minimizes cost, so this finds paths through actors who collaborated **least**.

In [ ]:
# Run Yen's with collaboration weights
yens_weighted = gds.shortestPath.yens.stream(
    G,
    sourceNode=source_id,
    targetNode=target_id,
    k=5,
    relationshipWeightProperty='collaborations'
)

print(yens_weighted)

And let's take a look at that in a more readable format:

In [ ]:
# Display detailed results
weighted_results = []
for idx, row in yens_weighted.iterrows():
    # Get actor names along the path
    path_names = gds.run_cypher("""
        UNWIND $nodeIds AS nodeId
        RETURN gds.util.asNode(nodeId).name AS name
    """, params={"nodeIds": list(row['nodeIds'])})
    
    weighted_results.append({
        'rank': row['index'] + 1,
        'total_cost': row['totalCost'],
        'path': ' -> '.join(path_names['name'].tolist()),
        'costs': list(row['costs']) if 'costs' in row else None
    })

weighted_df = pd.DataFrame(weighted_results)
print("Weighted Paths (minimizing collaboration count):")
print(weighted_df[['rank', 'total_cost', 'path']].to_string(index=False))

### Visualize: Weighted K-Shortest Paths

In [ ]:
# Visualize weighted paths
VG = visualize_k_paths(yens_weighted, "Top 5 Paths by Weakest Collaborations")
VG.render()

As with Dijkstra's, we can **invert** the weights to get the strongest collaboration paths. 

First, we can project a new graph into the same session we've already been using -- this won't disrupt the other graph in the session.

In [ ]:
# Project with inverted weights
G_inverted, result = gds.graph.project(
    "actor-inverted-weights",
    """
    CALL {
        MATCH (a1:Actor)-[:ACTED_IN]->(m:Movie)<-[:ACTED_IN]-(a2:Actor)
        WHERE a1 <> a2
        WITH a1, a2, count(m) AS collaborations
        RETURN a1 AS source, a2 AS target, 1.0 / collaborations AS invertedCollab
    }
    RETURN gds.graph.project.remote(
        source,
        target,
        {relationshipProperties: {invertedCollab: invertedCollab}}
    )
    """
)

print(f"Projected graph: {G_inverted.name()}")
print(f"  Nodes: {G_inverted.node_count():,}")
print(f"  Relationships: {G_inverted.relationship_count():,}")

Next, we just run Yen's as normal, using the inverted weights. 

In [ ]:
# Run Yen's with inverted weights (finds strongest collaboration paths)
yens_inverted = gds.shortestPath.yens.stream(
    G_inverted,
    sourceNode=source_id,
    targetNode=target_id,
    k=5,
    relationshipWeightProperty='invertedCollab'
)

print(yens_inverted)

Notice now how there is a lot more information held in the paths -- let's print a more readable format:

In [ ]:
# Display strongest collaboration paths
inverted_results = []
for idx, row in yens_inverted.iterrows():
    # Get actor names along the path
    path_names = gds.run_cypher("""
        UNWIND $nodeIds AS nodeId
        RETURN gds.util.asNode(nodeId).name AS name
    """, params={"nodeIds": list(row['nodeIds'])})
    
    inverted_results.append({
        'rank': row['index'] + 1,
        'total_cost': row['totalCost'],
        'path': ' -> '.join(path_names['name'].tolist())
    })

inverted_df = pd.DataFrame(inverted_results)
print("Strongest Collaboration Paths (inverted weights):")
print(inverted_df[['rank', 'total_cost', 'path']].to_string(index=False))

In the previous version, when we minimized collaboration counts, we got the same 4-hop cost for every path. That happened because every hop in our shortest path had only one collaboration count. 

However, in our inverted version, we can see that Shah Rukh Kahn has a high collaboration-count, and shorter path, through Deepika Padukone.

Let's see how offen each of our actors have collaborated along these paths. 

In [ ]:
# Get raw collaboration counts for each path
print("Raw Collaboration Counts Along Each Path:")
print("=" * 60)

for idx, row in yens_inverted.iterrows():
    # Get actor names along the path
    node_ids = list(row['nodeIds'])
    path_names = gds.run_cypher("""
        UNWIND $nodeIds AS nodeId
        RETURN gds.util.asNode(nodeId).name AS name
    """, params={"nodeIds": node_ids})['name'].tolist()
    
    # Get collaboration counts between consecutive pairs
    collab_counts = []
    for i in range(len(path_names) - 1):
        collab = gds.run_cypher("""
            MATCH (a1:Actor {name: $name1})-[:ACTED_IN]->(m:Movie)<-[:ACTED_IN]-(a2:Actor {name: $name2})
            RETURN count(m) AS collaborations
        """, params={"name1": path_names[i], "name2": path_names[i+1]})['collaborations'].iloc[0]
        collab_counts.append(collab)
    
    print(f"\nRank {row['index'] + 1} (inverted cost: {row['totalCost']:.3f}):")
    for i, (actor, collab) in enumerate(zip(path_names[:-1], collab_counts)):
        print(f"  {actor} --({collab} movies)--> {path_names[i+1]}")
    print(f"  Total collaborations: {sum(collab_counts)}")

While Yen's makes it all seem quite simple, the results demonstrate the level of complexity it can blast through. 

Shah Rukh Khan has acted in 5 movies with Deepika Padukone. However, Deepika has only acted in 1 movie with Donnie Yen -- the next on the list. 

There are of course many other paths between Deepika and Lee Jung-jae with a higher collaboration count -- but Yen's knows that subbing in this low-count hop will get us to Lee Jung-jae faster than other paths with stronger collaborations.

In [ ]:
# Visualize strongest collaboration paths
VG = visualize_k_paths(yens_inverted, "Top 5 Paths by Strongest Collaborations (Inverted Weights)")
VG.render()

The same algorithm with different inputs can give very different results. The "best" path depends entirely on what you're optimizing for. In logistics, this might be distance vs. cost vs. time.

## Performance Considerations

Yen's is more expensive than Dijkstra's:

- Runs Dijkstra's internally for _each_ of the k paths
- Cost scales roughly with k
- Still efficient for reasonable k values

## When Yen's Won't Help

Yen's is **not** the right choice when:

- You only need the single best path -> Use Dijkstra's
- You need paths from one source to **multiple** targets -> Use Dijkstra's with targetNodes list
- You need **all** shortest paths in the graph -> Use All Pairs Shortest Path

## Clean Up

In [ ]:
# Drop the projection
dropped_graphs = G.drop()
print(f"{dropped_graphs} dropped")

In [ ]:
# Delete the session
gds.delete()
print("Session deleted - billing stopped")

## From Movies to Logistics

You've now learned Yen's on the Movie graph—but everything transfers to other contexts:

| Movies | Logistics |
|--------|----------|
| Multiple paths between actors | Alternative shipping routes |
| Ranked by hops/collaborations | Ranked by transit time |
| Compare top k paths | Compare to historical route choices |

In the next lesson, you'll apply Yen's to the logistics network—and discover the cost of a world without Yen's.

## Summary

Yen's finds the k shortest paths between two nodes, enabling comparison and backup planning.

What we covered:

- Returns k paths ranked by total cost
- Builds on Dijkstra's—runs it multiple times internally
- Single target only—unlike Dijkstra's, no targetNodes list
- Respects parallel relationships—can find paths using different edges between same nodes
- Three execution modes—stream, mutate, and write

In the next lesson, you'll apply Yen's to the logistics network to find multiple routes and compare them to historical operations.